In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import datetime
import scipy.stats
#import plotly.offline as pyo
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as cl
import matplotlib.dates as mdates
import matplotlib
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import seaborn as sns
import netCDF4 as nc
import xarray as xr
#import cartopy.crs as ccrs
#from cartopy import feature as cfa
import os
import plotly.io as pio
pio.renderers.default = 'notebook'

In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    decimal_part = decimal_year - year
    start_of_year = pd.to_datetime([f'{y}-01-01' for y in year])
    seconds_in_year = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - start_of_year).total_seconds()
    seconds = decimal_part * seconds_in_year
    return start_of_year + pd.to_timedelta(seconds, unit='s')

def datetime_to_decimal_year(datetime_):
    year = datetime_.year
    days = (datetime_.date() - datetime.date(year, 1,1)).days
    res = year + days/365.25
    return res


def FFT_res(x, si):
    n = len(x)
    x_FFT_shifted = np.fft.fftshift(np.fft.fft(x))
    x_FFT_freq_shifted = np.fft.fftshift(np.fft.fftfreq(n, d=si))
    return x_FFT_freq_shifted, x_FFT_shifted


def fft_noramlize(fft_res):
    n = len(fft_res)
    norm_fft = np.abs(fft_res)/(np.max(np.abs(fft_res)))
    return norm_fft

def line_fit(X,Y):
    P = np.polyfit(X,Y,1)
    dx = np.max(X) - np.min(X)
    X_new = np.array([np.min(X)-dx*0.05, np.max(X)+dx*0.05])
    Y_new = np.polyval(P, X_new)
    return X_new, Y_new, P

In [ ]:
%run ../20260323_read_data.py
df_sorted

In [ ]:
df_sorted['age_final_month'] = df_sorted['age_dt'].dt.month
df_updated_monthly = df_sorted[['age_final_month', 'd18o_pdb', 'd18o', 'd13c']].groupby('age_final_month').mean()
df_updated_monthly

In [ ]:
morpho_data = pd.read_csv('../../../2026-01-08_Luc-Morphology/SeaLevelFromMorphology.csv', header=None)
cols = ['date', 'SPA_luc', 'SPLA_luc','SPA_fred', 'SPLA_fred']
morpho_data.columns = cols
morpho_data

morpho_raw = pd.read_excel('../20260306_agemodel/Morphology_Data_ageupdated.xlsx')
#scale_factor_y = 10./(57.69-75.96)
scale_factor_y = (-10.--90.)/(3.03-149.03)
scale_factor_x = -scale_factor_y
offset_y =  -60-scale_factor_y*(94.27)
morpho_raw.loc[:,'x_true'] = morpho_raw.loc[:,'x']*scale_factor_x
morpho_raw.loc[:,'y_true'] = morpho_raw.loc[:,'y']*scale_factor_y + offset_y
morpho_memo = morpho_raw.loc[~morpho_raw['x'].isna(),:].sort_values("age_absolute")
morpho_memo

In [ ]:
window_size1=12

fred = pd.read_excel('../20260309_freds-data/fred_age_model_v2.xlsx').dropna()
fred['d18o_pdb'] = -(fred['d18o']-20.91)/1.03091
fred[['age_ma','d13c_ma','d18o_ma', 'd18o_pdb_ma']] = fred[['calendar_year','d13c','d18o','d18o_pdb']].rolling(window_size, center=True).mean()


cols = ['calendar_year','d13c','d18o','d18o_pdb']
new_cols = ['age_ma2','d13c_ma2','d18o_ma2','d18o_pdb_ma2']
rename_dict = dict(zip(cols,new_cols))
right_roll = fred[cols].rolling(window_size1).mean().rename(columns=rename_dict)
left_roll = fred[cols].rolling(window_size1,center=True).mean().shift(window_size1).rename(columns=rename_dict)
fred.loc[:89, new_cols] = right_roll.loc[:89].to_numpy()
fred.loc[90:, new_cols] = left_roll.loc[90:].to_numpy()

fred[['d13c_std','d18o_std']] = fred[['d13c','d18o_pdb']].rolling(10, center=True).std()
fred_resampled = fred.drop(fred.columns[4:], axis=1)
fred_resampled['age_dt'] = decimal_year_to_datetime(fred_resampled['calendar_year'])
fred_resampled.set_index('age_dt', inplace=True)
fred_resampled = fred_resampled.resample('30D').mean()
fred_resampled.interpolate(inplace=True)
fred

In [ ]:
climate = pd.read_csv('../../../2026-01-15_Climate-Model/analysis/climate_df.csv')
climate['datetime'] = pd.to_datetime(climate['datetime'])
climate_month = climate.groupby(climate['datetime'].dt.month).mean()
climate_month

In [ ]:
tide = pd.read_csv('../../../2025-10-10_Fred-TideGauge/memo_edited.csv')
tide['age_dt'] = decimal_year_to_datetime(tide['Date'])#.dt.date
tide = tide.set_index('age_dt')
tide

In [ ]:
enso_phase = pd.read_csv('../../../ENSO_Phase_Classification.csv')
enso_phase.columns=['Year','Phase']
enso_phase['Date'] = decimal_year_to_datetime(enso_phase['Year'])
#enso_phase.set_index('Date', inplace=True)
enso_phase['Phase'] = enso_phase['Phase'].str.replace('Warm Phase', 'W').str.replace('Cold Phase', 'C').str.replace('Neutral Phase','N')


enso = pd.read_csv("../../../ENSO_Time_Series.csv", parse_dates=[0])
#enso = enso.set_index("Date")
enso['day_of_year'] = enso['Date'].dt.dayofyear
enso['is_leap'] = enso['Date'].dt.is_leap_year
enso['total_days'] = np.where(enso['is_leap'], 366, 365)
enso['decimal_year'] = enso['Date'].dt.year + (enso['day_of_year'] - 1) / enso['total_days']


margin = 0.5
enso['phase'] = 'N'
enso.loc[enso['ENSO']>=margin,'phase'] = "C"
enso.loc[enso['ENSO']<=-margin,'phase'] = "W"
enso['phase_group'] = (enso['phase'] != enso['phase'].shift()).cumsum()
blocks = enso.groupby('phase_group').agg(
    start=('Date', 'first'),
    end=('Date', 'last'),
    value=('phase', 'first')
).reset_index(drop=True)

enso['Date'] = enso['Date'] + pd.offsets.MonthEnd(0)
enso.set_index('Date', inplace=True, drop=False)

ENSO_match = np.array([[1982.2, 1984.1],
                       [1986.75, 1988.3],
                       [1992.1, 1992.96],
                       [1997.2, 1998.8]])

enso_fft_freq, enso_fft = FFT_res(enso['ENSO'], 30)


enso


In [ ]:
fig, ax = plt.subplots(1, figsize=(6,4))


ax.plot(np.arange(1, 13), climate_month['synthetic_d18o'], '--o',color='blue',label='d18o climate')
ax.plot(np.arange(1, 13), df_updated_monthly['d18o_pdb']*2+10.2, '-^', color='steelblue', label='d18o obs')
ax.legend(loc='lower right')

ax.set_xlabel('Month of Year')
ax.set_ylabel(r'$\delta^{18}$O')
ax.set_xticks(np.arange(1, 13))


ax2 = ax.twinx()
#ax2.plot(climate_monthly['d13c'], color='green')
ax2.plot(np.arange(1, 13), df_updated_monthly['d13c'],  '-^',color='green',label='d13c obs')
#ax2.plot(df_updated_monthly['d18o_pdb']/df_updated_monthly['d18o_pdb'].max(), '--', color='blue', label='model')
ax2.set_ylabel(r'$\delta^{13}$C')
ax2.legend(loc='upper left')

fig.savefig('monthly_climate_model.pdf')

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(16,4))

ax = axes[0]
ax.plot(np.arange(1, 13), climate_month['synthetic_d18o'], '--o',color='steelblue', label=r'synthetic $\delta^{18}$O')
ax.plot(np.arange(1, 13), df_updated_monthly['d18o_pdb']*2+10.2, '-^', color='blue', label='d18o obs')
ax.set_ylabel(r'$\delta^{18}$O')
ax.set_xlabel('Months')
ax.set_xticks(np.arange(1,13))
ax.set_title(r'Climate Average for $\delta^{18}$O')
ax.legend()
#fig.savefig('climate_average.pdf')
ax.set_xticks(np.arange(1, 13))

ax = axes[1]
ax.plot(np.arange(1, 13), climate_month['SSS'], '^-', color='#46bd9b', label='SSS')
ax2 = ax.twinx()
ax2.plot(np.arange(1, 13), climate_month['SST'], 'v-', color='#F54927', label='SST')
ax.set_xticks(np.arange(1, 13))
ax.legend(loc='lower right')
ax2.legend(loc='lower left')
ax.set_xlabel('Months')


ax = axes[2]
#ax2 = ax.twinx()
#ax2.plot(climate_monthly['d13c'], color='green')
ax.plot(np.arange(1, 13), df_updated_monthly['d13c'],  '-^',color='green',label='d13c obs')
#ax2.plot(df_updated_monthly['d18o_pdb']/df_updated_monthly['d18o_pdb'].max(), '--', color='blue', label='model')
ax.set_ylabel(r'$\delta^{13}$C')
ax.legend(loc='upper left')
ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")
ax.set_xlabel('Months')
ax.set_xticks(np.arange(1, 13))

fig.savefig('climate_isotpe_monthly.png')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), subplot_kw={'projection': 'polar'})

ax = axes[0]

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

theta = np.linspace(0, 2*np.pi, 12, endpoint=False)
width = 2*np.pi/12 * 0.9  # 90% of bin width (small gap)

rmin, rmax = -0.21 , 0.3
vals = climate_month['synthetic_d18o'].values

# Make bars start at rmin, and only show the variation within [rmin, rmax]
heights = np.clip(vals, rmin, rmax) - rmin

ax.bar(theta, heights, width=width, bottom=rmin,
       color='#1A388A', alpha=0.7, edgecolor='black', linewidth=1.5)

ax.set_rlim(rmin, rmax)

ax.set_rticks([])
ax.set_xticks(theta)
ax.set_xticklabels(months)

ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)

ax.set_title(r'Climate based $\delta^{18}$O')

ax = axes[1]

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

theta = np.linspace(0, 2*np.pi, 12, endpoint=False)
width = 2*np.pi/12 * 0.9  # 90% of bin width (small gap)

rmin, rmax = -5.2, -4.95
vals = df_updated_monthly['d18o_pdb'].values

# Make bars start at rmin, and only show the variation within [rmin, rmax]
heights = np.clip(vals, rmin, rmax) - rmin

ax.bar(theta, heights, width=width, bottom=rmin,
       color='steelblue', alpha=0.7, edgecolor='black', linewidth=1.5)

ax.set_rlim(rmin, rmax)

ax.set_rticks([])
ax.set_xticks(theta)
ax.set_xticklabels(months)

ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)

ax.set_title(r'Observed $\delta^{18}$O')


ax = axes[2]


rmin, rmax = -1.45, -1.18
vals = df_updated_monthly['d13c'].values

# Make bars start at rmin, and only show the variation within [rmin, rmax]
heights = np.clip(vals, rmin, rmax) - rmin

ax.bar(theta, heights, width=width, bottom=rmin,
       color='green', alpha=0.7, edgecolor='black', linewidth=1.5)

ax.set_rlim(rmin, rmax)

ax.set_rticks([])
ax.set_xticks(theta)
ax.set_xticklabels(months)

ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)

ax.set_title(r'Observed $\delta^{13}$C')

plt.suptitle('Matching Peak Age Model', y= 1.05 , fontsize=16)


#fig.savefig('d13c_d18o_monthly_means.pdf',bbox_inches='tight' )
#fig.savefig('d13c_d18o_monthly_means.png',bbox_inches='tight' )

In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])

#fig.add_trace(go.Scatter(x=df_sorted['age_dt'], y=df_sorted['d18o']-df_sorted['d18o'].mean(),  mode="markers", marker_color="#3492eb",  name='d18O', text=df_sorted['Name']),secondary_y=False)
#fig.add_trace(go.Scatter(x=df_resampled.index.values, y=df_resampled['d18o_ma']-df_resampled['d18o_ma'].mean(), mode="lines", line_color='#134b80', name='average'), secondary_y=False)
#fig.add_trace(go.Scatter(x=df_resampled.index.values, y=df_resampled['d18o_ma2']-df_resampled['d18o_ma2'].mean(), mode="lines", line_color='navy', name='average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=df_sorted['age_dt'], y=df_sorted['d18o_no_tectonic'] ,  mode="markers", marker_color="#3492eb",  name='d18O', text=df_sorted['Name']),secondary_y=False)
#fig.add_trace(go.Scatter(x=df_sorted['age_dt'], y=df_sorted['d18o_notec_ma'] , mode="lines", line_color='#134b80', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d18o'] , mode="markers", line_color='#3492eb',text=df_sorted['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d18o_ma'] , mode="lines", line_color='#134b80', name='average'), secondary_y=False)



fig.add_trace(go.Scatter(x=climate['age_absolute'], y= climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1900, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O-AGEMODELTEST.html')


In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c'] , mode="markers", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'] , mode="lines", line_color='#05700C', name='3 month average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d13c_ma2'] , mode="lines", line_color='#011803', name='3 year average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)
fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)
#fig.add_trace(go.Scatter(x=enso['decimal_year'], y= enso['ENSO'] , mode="lines", line_color='gray', name='Sea Level', line_width=2), secondary_y=True)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1700, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{13}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= 'Sea Level [mm]', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O.html')


In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c'] , mode="markers", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'] , mode="lines", line_color='#05700C', name='3 month average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d13c_ma2'] , mode="lines", line_color='#011803', name='3 year average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)
fig.add_trace(go.Scatter(x=enso['decimal_year'], y= enso['ENSO'] , mode="lines", line_color='gray', name='Sea Level', line_width=2), secondary_y=True)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1700, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{13}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= 'Sea Level [mm]', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O.html')


In [ ]:
df_resampled

In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c'] , mode="markers", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c'] , mode="lines", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'] , mode="lines", line_color='#05700C', name='3 month average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d13c_ma2'] , mode="lines", line_color='#011803', name='3 year average'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_resampled['age_resampled'], y=df_resampled['d13c'] , mode="markers", line_color='#C20A20',text=df_sorted['Name'], name='raw'), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

#fig.update_layout(width=1700, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
fig.update_layout(width=1000, height=500, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')

# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O.html')


In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c_no_tectonic'] , mode="lines", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)
#fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c'] , mode="lines", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)

#fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'] , mode="lines", line_color='#05700C', name='3 month average'), secondary_y=False)
#fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d13c_ma2'] , mode="lines", line_color='#011803', name='3 year average'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_resampled['age_resampled'], y=df_resampled['d13c_no_tectonic'] , mode="markers", line_color='#C20A20',text=df_sorted['Name'], name='raw'), secondary_y=False)


fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

#fig.update_layout(width=1700, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
fig.update_layout(width=1000, height=500, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')

# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O.html')


In [ ]:


fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d18o'] , mode="markers", line_color='#3492eb',text=df_sorted['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d18o'] , mode="lines", line_color='#3492eb',text=df_sorted['Name'], name='raw'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d18o_ma'] , mode="lines", line_color='#134b80', name='3 month average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d18o_ma2'] , mode="lines", line_color='#0D0454', name='3 year average'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_resampled['age_resampled'], y=df_resampled['d18o'] , mode="markers", line_color='#C20A20',text=df_sorted['Name'], name='raw'), secondary_y=False)

fig.add_trace(go.Scatter(x=aligned['age_absolute'], y=aligned['d18o_no_tec_sg']+25.75 , mode="lines", line_color='green',text=df_sorted['Name'], name='aligned_d18o'), secondary_y=True)



#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=2), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1700, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O.html')


In [ ]:
df_resampled['d18o']

In [ ]:
# ── ALIGNED: interpolate all external series to exact coral sample ages ─────
#
# WHY THE OLD APPROACH BROKE
# ---------------------------
# The previous code resampled coral to a 3ME grid, then joined ENSO / tide /
# climate via separate .reindex().interpolate().loc[target_dates] calls.
# Each join used a different intermediate index so after dropna() the surviving
# rows differed column by column — rolling().corr() then compared values at
# DIFFERENT calendar times, giving meaningless correlations.
#
# THE FIX
# -------
# Keep df_sorted at its exact irregular sample ages (decimal years).
# Map every external series onto those same ages with a single linear
# interpolator.  All columns share one index → rolling().corr() is correct.

from scipy.signal import savgol_filter
from scipy.interpolate import interp1d

def interp_to(x_ref, y_ref, target):
    """Piecewise-linear interpolation; returns NaN outside x_ref range."""
    x_ref  = np.asarray(x_ref,  float)
    y_ref  = np.asarray(y_ref,  float)
    target = np.asarray(target, float)
    out    = np.full(len(target), np.nan)
    for i, a in enumerate(target):
        idx = np.searchsorted(x_ref, a)
        if 0 < idx < len(x_ref):
            t      = (a - x_ref[idx-1]) / (x_ref[idx] - x_ref[idx-1])
            out[i] = y_ref[idx-1] + t * (y_ref[idx] - y_ref[idx-1])
    return out

# Use exact sample ages as the shared axis
ages = df_sorted['age_final'].values

# ── Savitzky-Golay tectonic detrend (matches read_data.py) ────────────────────
# Build a monthly grid, apply SG (73-month window, order 3), interpolate back.
def to_monthly(a): return (np.array(a) * 12).round() / 12

df_sorted_copy = df_sorted.copy()
df_sorted_copy['mb'] = to_monthly(ages)
coral_mon = df_sorted_copy.groupby('mb')[['d13c', 'd18o_pdb']].mean().reset_index()
coral_mon.rename(columns={'mb': 'age'}, inplace=True)

for col in ['d13c', 'd18o_pdb']:
    sg = savgol_filter(
        coral_mon[col].interpolate().ffill().bfill().values,
        window_length=73, polyorder=3, mode='interp'
    )
    fn = interp1d(coral_mon['age'].values, sg,
                  bounds_error=False, fill_value='extrapolate')
    df_sorted[f'{col}_sg_trend']  = fn(ages)
    df_sorted[f'{col}_no_tec_sg'] = df_sorted[col] - df_sorted[f'{col}_sg_trend']

# ── Climate ────────────────────────────────────────────────────────────────────
clim_s = climate.sort_values('age_absolute')
for col in ['SST', 'SSS', 'synthetic_d18o']:
    df_sorted[col] = interp_to(clim_s['age_absolute'].values, clim_s[col].values, ages)

# ── Tide gauge ────────────────────────────────────────────────────────────────
tide_s  = tide.dropna(subset=['Sea Level']).sort_values('Date')
sl_raw  = interp_to(tide_s['Date'].values, tide_s['Sea Level'].values, ages)
df_sorted['Sea Level'] = (sl_raw - np.nanmean(sl_raw)) / 1000.0   # mm → m, remove mean

# ── ENSO ──────────────────────────────────────────────────────────────────────
enso_dt  = pd.to_datetime(enso['Date'])
enso_age = (enso_dt.dt.year + (enso_dt.dt.month - 0.5) / 12).values
order    = np.argsort(enso_age)
df_sorted['ENSO'] = interp_to(enso_age[order], enso['ENSO'].values[order], ages)

# ── Build aligned: single float index, one row per coral sample ──────────────
aligned = df_sorted[[
    'age_final',
    'd13c', 'd18o', 'd18o_pdb',
    'd13c_no_tec_sg', 'd18o_pdb_no_tec_sg',
    'SST', 'SSS', 'synthetic_d18o',
    'Sea Level', 'ENSO'
]].rename(columns={'age_final': 'age'}).set_index('age').sort_index()

print(f"aligned: {len(aligned)} rows  {aligned.index.min():.2f}–{aligned.index.max():.2f}")
print(f"  SST:       {aligned['SST'].notna().sum()} samples")
print(f"  Sea Level: {aligned['Sea Level'].notna().sum()} samples")
print(f"  ENSO:      {aligned['ENSO'].notna().sum()} samples")
aligned


In [ ]:
import seaborn as sns

correlation_matrix = aligned.corr()

fig = plt.figure(figsize=(9, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt=".2f")
plt.title("Correlation Matrix — aligned dataset")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib as mpl

# ENSO event windows for shading
_ENSO_WINDOWS = [
    (1982 + 2/12,  1984 + 1/12,  '#ff6b35'),   # El Niño
    (1986 + 9/12,  1988 + 1/12,  '#ff6b35'),
    (1992 + 1/12,  1993 + 1/12,  '#ff6b35'),
    (1997 + 2/12,  1998 + 10/12, '#ff6b35'),
    (1984 + 6/12,  1985 + 6/12,  '#00bfff'),   # La Niña
    (1988 + 6/12,  1989 + 6/12,  '#00bfff'),
    (1998 + 10/12, 2000 + 3/12,  '#00bfff'),
]

def plot_rolling_corr(df, proxy, drivers, win,
                      proxy_col='#27F535', driver_cols=None,
                      xlim=(1958, 2007), save_path=None):
    """
    Plot rolling correlation between `proxy` and each series in `drivers`.

    Parameters
    ----------
    df          : aligned DataFrame with float (decimal-year) index
    proxy       : column name of the coral proxy (e.g. 'd13c_no_tec_sg')
    drivers     : list of (col_name, label, colour) tuples
    win         : rolling window size in samples
    proxy_col   : colour for the proxy trace
    xlim        : (xmin, xmax) for the x-axis
    save_path   : if given, save figure to this path
    """
    if driver_cols is None:
        driver_cols = ['#4ecdc4', '#e05c5c', '#aaaaaa']

    n      = len(drivers)
    fig, axes = plt.subplots(
        n * 2, 1, figsize=(18, 4 * n),
        sharex=True,
        gridspec_kw={'hspace': 0,
                     'height_ratios': [2, 1] * n}
    )

    cmap_rc = mpl.cm.RdBu_r
    norm_rc  = mpl.colors.Normalize(vmin=-1, vmax=1)

    x = df.index.values

    for pi, (drv_col, drv_lbl, dc) in enumerate(drivers):
        ax_sig = axes[pi * 2]
        ax_cor = axes[pi * 2 + 1]

        for ax in (ax_sig, ax_cor):
            ax.set_facecolor('#111720')
            for sp in ax.spines.values():
                sp.set_color('#1e2a38')
            ax.tick_params(colors='#5a7090', labelsize=8.5)
            ax.grid(color='#1a2535', lw=0.5, alpha=0.6)

        p  = df[proxy].values
        dr = df[drv_col].values

        # Signal panel
        ax_sig.plot(x, p, color=proxy_col, lw=0.8, alpha=0.35)
        ax_sig.plot(x,
                    pd.Series(p).rolling(8, center=True, min_periods=3).mean().values,
                    color=proxy_col, lw=2.2)
        ax_sig.set_ylabel(proxy, color=proxy_col, fontsize=8.5)
        ax_sig.tick_params(axis='y', labelcolor=proxy_col)

        ax2 = ax_sig.twinx()
        ax2.set_facecolor('#111720')
        valid_d = ~np.isnan(dr)
        ax2.plot(x[valid_d], dr[valid_d], color=dc, lw=1.5, alpha=0.85)
        ax2.set_ylabel(drv_lbl, color=dc, fontsize=8.5)
        ax2.tick_params(axis='y', labelcolor=dc)
        for sp in ax2.spines.values():
            sp.set_color('#1e2a38')

        # Rolling correlation
        rc = df[proxy].rolling(win, center=True, min_periods=win // 2).corr(df[drv_col])
        y     = rc.values
        valid = ~np.isnan(y)
        xv    = x[valid]; yv = y[valid]

        ax_cor.plot(x, y, color='white', lw=1.0, zorder=3)
        ax_cor.axhline(0, color='#5a7090', lw=0.8, ls='--')
        for i in range(len(xv) - 1):
            ax_cor.axvspan(xv[i], xv[i+1],
                           color=cmap_rc(norm_rc(yv[i])), alpha=0.65, linewidth=0)

        n_eff  = int(df[drv_col].notna().sum())
        thresh = 1.97 / np.sqrt(max(min(win, n_eff) - 2, 2))
        ax_cor.axhline( thresh, color='white', lw=0.9, ls=':', alpha=0.5,
                        label=f'p=0.05 (±{thresh:.2f})')
        ax_cor.axhline(-thresh, color='white', lw=0.9, ls=':', alpha=0.5)
        ax_cor.set_ylim(-1, 1)
        ax_cor.set_ylabel('r', color='#8a9bb0', fontsize=8.5)
        if pi == 0:
            ax_cor.legend(fontsize=7.5, facecolor='#111720',
                          edgecolor='#1e2a38', labelcolor='#cdd6e0', loc='upper left')

        # ENSO shading
        for t0, t1, ec in _ENSO_WINDOWS:
            ax_sig.axvspan(t0, t1, alpha=0.08, color=ec, zorder=0)
            ax_cor.axvspan(t0, t1, alpha=0.08, color=ec, zorder=0)

    axes[-1].set_xlabel('Calendar year', color='#8a9bb0', fontsize=9.5)
    for ax in axes:
        ax.set_xlim(xlim)

    axes[0].set_title(
        f'Rolling Correlation — {proxy}  |  window = {win} samples ≈ {win/8:.1f} yr  |  '        f'orange = El Niño  |  blue = La Niña  |  dotted = p=0.05',
        color='#cdd6e0', fontsize=11, pad=8
    )
    fig.patch.set_facecolor('#0a0e14')

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='#0a0e14')
        print(f"Saved: {save_path}")
    plt.show()

print("plot_rolling_corr() ready.")


In [ ]:
# δ¹³C_no_tec_sg vs climate / sea level / ENSO  — two window sizes
drivers_d13c = [
    ('synthetic_d18o', r'Climate δ¹⁸O',  '#4ecdc4'),
    ('Sea Level',      'Sea Level (m)',   '#e05c5c'),
    ('ENSO',          'ENSO index',       '#aaaaaa'),
]

for win in [16, 36]:
    plot_rolling_corr(aligned, 'd13c_no_tec_sg', drivers_d13c, win=win,
                      save_path=f'correlation_d13c_win{win}.png')


In [ ]:
# δ¹⁸O_no_tec_sg vs climate / sea level / ENSO  — two window sizes
drivers_d18o = [
    ('synthetic_d18o', r'Climate δ¹⁸O',  '#4ecdc4'),
    ('Sea Level',      'Sea Level (m)',   '#e05c5c'),
    ('ENSO',          'ENSO index',       '#aaaaaa'),
]

for win in [16, 36]:
    plot_rolling_corr(aligned, 'd18o_pdb_no_tec_sg', drivers_d18o, win=win,
                      proxy_col='#3492eb',
                      save_path=f'correlation_d18o_win{win}.png')


In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c'] , mode="markers", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'] , mode="lines", line_color='#05700C', name='3 month average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d13c_ma2'] , mode="lines", line_color='#011803', name='3 year average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)
fig.add_trace(go.Scatter(x=morpho_memo['age_absolute'], y=-morpho_memo['y_true'], mode="lines", line=dict(color='purple', width=4), name='SPLA morpho memo'), secondary_y=True)


fig.add_trace(go.Scatter(x=morpho_data['date']-0.5, y=-morpho_data['SPLA_luc'], mode="lines", line=dict(color='firebrick', width=4), name='SPLA morpho luc'), secondary_y=True)
fig.add_trace(go.Scatter(x=morpho_data['date']-0.5, y=morpho_data['SPLA_fred'], mode="lines", line=dict(color='orange', width=4), name='SPLA morpho fred'), secondary_y=True)

fig.add_trace(go.Scatter(x=morpho_data['date']-0.5, y=-morpho_data['SPA_luc'], mode="lines", line=dict(color='navy', width=4), name='SPA morpho luc'), secondary_y=True)
fig.add_trace(go.Scatter(x=morpho_data['date']-0.5, y=-morpho_data['SPA_fred'], mode="lines", line=dict(color='black', width=4), name='SPA morpho fred'), secondary_y=True)




fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1600, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{13}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= 'Sea Level [mm]', secondary_y=True)

fig.show()
#fig.write_html('d13c_all-morphologies.html')


In [ ]:
fred

In [ ]:
 fred.loc[90:,['calendar_year','d13c','d18o','d18o_pdb']].rolling(window_size1, closed='left').mean().shift(-(window_size1 ))

In [ ]:
# assign
fred.loc[:89, ['age_ma2','d13c_ma2','d18o_ma2','d18o_pdb_ma2']] = right_roll.loc[:89].to_numpy()
fred.loc[90:, ['age_ma2','d13c_ma2','d18o_ma2','d18o_pdb_ma2']] = left_roll.loc[90:].to_numpy()

In [ ]:
#fred[fred['decimal_year']<2007.25]
x = pd.concat([fred.loc[:89,['d13c','d18o','calendar_year','d18o_pdb']].rolling(window_size1, center=True).mean(),
         fred.loc[90:,['d13c','d18o','calendar_year','d18o_pdb']].rolling(window_size1).mean().shift(-(window_size1//2 ))])


x.head(20)

In [ ]:
window_size1 = 16
split = 2007.25

# split masks
before = fred["calendar_year"] <= split
after = fred["calendar_year"] > split

# rolling means
right_roll = fred["d13c"].rolling(window_size1).mean()
left_roll = fred["d13c"].rolling(window_size1).mean().shift(-(window_size1-1))

ma = pd.Series(index=fred.index, dtype=float)
ma.loc[before] = right_roll.loc[before]
ma.loc[after] = left_roll.loc[after]

In [ ]:

ma

In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=fred['calendar_year'], y=fred['d13c'] , mode="markers", line_color='#27F535', name='fred d13c'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['age_ma'], y=fred['d13c_ma'] , mode="lines", line_color='#05700C', name='moving average'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['calendar_year'], y=ma , mode="lines", line_color='#011803', name='moving average 2'), secondary_y=False)


fig.add_trace(go.Scatter(x=fred['calendar_year'], y=fred['d18o'] , mode="markers", line_color='#3492eb', name='fred d18O'), secondary_y=True)
fig.add_trace(go.Scatter(x=fred['age_ma'], y=fred['d18o_ma'] , mode="lines", line_color='#134b80', name='moving average'), secondary_y=True)

fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1600, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{13}$ C Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=True)


fig.show()
#fig.write_html('d13c_all-morphologies.html')


In [ ]:
window_size1=12

fred_old = pd.read_excel('../../../2025-08-22_FredsData-post2008/FredData_editedMemo_raw.xlsx').dropna()
fred_old['d18o_pdb'] = -(fred['d18o']-20.91)/1.03091
fred_old[['age_ma','d13c_ma','d18o_ma', 'd18o_pdb_ma']] = fred[['calendar_year','d13c','d18o','d18o_pdb']].rolling(window_size, center=True).mean()
fred_old

In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=fred['calendar_year'], y=fred['d13c'] , mode="markers", line_color='#27F535', name='fred d13c'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['age_ma'], y=fred['d13c_ma'] , mode="lines", line_color='#05700C', name='moving average'), secondary_y=False)
#fig.add_trace(go.Scatter(x=fred['calendar_year'], y=ma , mode="lines", line_color='#011803', name='moving average 2'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred_old['decimal_year'], y=fred['d13c'] , mode="markers", line_color='#F54522', name='old age model'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred_old['age_ma'], y=fred['d13c_ma'] , mode="lines", line_color='#872109', name='moving average'), secondary_y=False)



#fig.add_trace(go.Scatter(x=fred['calendar_year'], y=fred['d18o'] , mode="markers", line_color='#3492eb', name='fred d18O'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['age_ma'], y=fred['d18o_ma'] , mode="lines", line_color='#134b80', name='moving average'), secondary_y=True)

fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1600, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{13}$ C Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=True)


fig.show()
#fig.write_html('d13c_all-morphologies.html')
